#Paquetes necesarios

In [1]:
import cv2  
import math 

from ultralytics import YOLO

Modelos preentrenados, visualizando con las utilidades de ultralytics

In [2]:
# Carga del modelo
#model = YOLO('yolo11n.pt') #Contenedores
#model = YOLO('yolo11n-seg.pt') #Máscaras
model = YOLO('yolo11n-pose.pt')  #Pose

#Para un vídeo 
filename = "TGC23_PdH_C0056cut.mp4"
results = model(filename, show=True)

cv2.destroyAllWindows()


WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/375) c:\Users\ivanl\Desktop\Cosas de la uni\Año 2026 2027\VC\P1\P4\TGC23_PdH_C0056cut.mp4: 384x640 (no detections), 127.0ms
video 1/1 (frame 2/375) c:\Users\ivanl\Desktop\Cosas de la uni\Año 2026 2027\VC\P1\P4\TGC23_PdH_C0056cut.mp4: 384x640 (no detections), 66.6ms
video 1/1 (frame 3/375) c:\Users\ivanl\Desktop\Cosas de la uni\Año 2026 2027\VC\P1\P4\TGC23_PdH_C0056cut.mp4: 384x640 (no detections), 61.3ms
video 1/1 (frame 4/375) c:\Users\iv

Desde cámara, detección con yolo11, modelo nano. Visualización propia con OpenCV

In [3]:
# Carga del modelo, descarga en disco si no está presente en la carpeta
model = YOLO('yolo11n.pt') #Contenedores

# Etiqueta de las distintas clases
classNames = ["person", "bicycle", "car", "motorbike", "aeroplane", "bus", "train", "truck", "boat",
              "traffic light", "fire hydrant", "stop sign", "parking meter", "bench", "bird", "cat",
              "dog", "horse", "sheep", "cow", "elephant", "bear", "zebra", "giraffe", "backpack", "umbrella",
              "handbag", "tie", "suitcase", "frisbee", "skis", "snowboard", "sports ball", "kite", "baseball bat",
              "baseball glove", "skateboard", "surfboard", "tennis racket", "bottle", "wine glass", "cup",
              "fork", "knife", "spoon", "bowl", "banana", "apple", "sandwich", "orange", "broccoli",
              "carrot", "hot dog", "pizza", "donut", "cake", "chair", "sofa", "pottedplant", "bed",
              "diningtable", "toilet", "tvmonitor", "laptop", "mouse", "remote", "keyboard", "cell phone",
              "microwave", "oven", "toaster", "sink", "refrigerator", "book", "clock", "vase", "scissors",
              "teddy bear", "hair drier", "toothbrush"
              ]


# Captura desde la webcam
vid = cv2.VideoCapture(0)
  
while(True):      
    # fotograma a fotograma
    ret, img = vid.read()
  
    # si hay imagen válida
    if ret:  
        # Detecta en la imagen
        results = model(img, stream=True)
        
        # Para cada detección
        for r in results:
            boxes = r.boxes

            for box in boxes:
                # Contenedor
                x1, y1, x2, y2 = box.xyxy[0]
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2) # convert to int values
                
                # Confianza
                confidence = math.ceil((box.conf[0]*100))/100
                print("Confianza --->",confidence)

                # Clase
                cls = int(box.cls[0])
                print("Clase -->", classNames[cls])

                # Convierte identificador numérico de clase a un color RGB
                escala = int((cls / len(classNames)) * 255 * 3)
                if escala >= 255*2:
                    R = 255
                    G = 255
                    B = escala - 255*2
                else:
                    if escala >= 255:
                        R = 255
                        G = escala - 255
                        B = 0
                    else:
                        R = escala
                        G = 0
                        B = 0

                # Dibuja el contenedor y clase
                cv2.rectangle(img, (x1, y1), (x2, y2), (R, G, B), 3)
                cv2.putText(img, classNames[cls] , [x1, y1], cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, B), 2)

        # Muestra fotograma
        cv2.imshow('Vid', img)
    
    # Detenemos pulsado ESC
    if cv2.waitKey(20) == 27:
        break
  
# Libera el objeto de captura
vid.release()
# Destruye ventanas
cv2.destroyAllWindows()


0: 480x640 (no detections), 101.6ms
Speed: 3.7ms preprocess, 101.6ms inference, 0.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 59.8ms
Speed: 3.0ms preprocess, 59.8ms inference, 0.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 67.7ms
Speed: 2.5ms preprocess, 67.7ms inference, 0.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 56.2ms
Speed: 2.1ms preprocess, 56.2ms inference, 0.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 55.7ms
Speed: 1.9ms preprocess, 55.7ms inference, 0.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 55.7ms
Speed: 2.1ms preprocess, 55.7ms inference, 0.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 57.3ms
Speed: 2.5ms preprocess, 57.3ms inference, 0.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 60.1ms
Speed: 2.4ms preprocess, 60.1ms

Seguimiento. Requiere instalar lap con pip install lap

In [ ]:
from collections import defaultdict
import numpy as np

# Carga del modelo, descarga en disco si no está presente en la carpeta
model = YOLO('yolo11n.pt') #Contenedores

# Etiqueta de las distintas clases
classNames = {0: "person", 1: "bicycle", 2: "car", 3: "motorbike", 5: "bus", 7: "truck"}


# Captura desde la webcam
vid = cv2.VideoCapture(0)
track_history = defaultdict(lambda: [])
  
while(True):      
    # fotograma a fotograma
    ret, img = vid.read()
  
    # si hay imagen válida
    if ret:  
        # Seguimiento, con persistencia entre fotogramas
        results = model.track(img, persist=True, classes = [0,1,2])

        if 0:
            if results is not None:
                print(results[0])
                boxes = results[0].boxes.xywh.cpu()
                track_ids = results[0].boxes.id.int().cpu().tolist()
                annotated_frame = results[0].plot()
                for box, track_id in zip(boxes, track_ids):
                    x, y, w, h = box
                    track = track_history[track_id]
                    track.append((float(x), float(y)))
                    if len(track) > 30:
                        track.pop(0)
                    points = np.hstack(track).astype(np.int32).reshape((-1, 1, 2))
                    cv2.polylines(annotated_frame, [points], isClosed=False, color=(230, 230, 230), thickness=10)
                cv2.imshow("YOLO11 Tracking", annotated_frame)
                if cv2.waitKey(1) & 0xFF == ord("q"):
                    break
        

        
        # Para cada detección
        for r in results:
            boxes = r.boxes

            for box in boxes:
                # Contenedor
                x1, y1, x2, y2 = box.xyxy[0]
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2) # convert to int values

                #Etiqueta de seguimiento
                if box.id is not None:
                    track_id = str(int(box.id[0].tolist()))
                else:
                    track_id = ''
                
                # Confianza
                confidence = math.ceil((box.conf[0]*100))/100
                print("Confianza --->",confidence)

                # Clase
                cls = int(box.cls[0])
                print("Clase -->", classNames[cls])

                # Convierte identificador numérico de clase a un color RGB
                escala = int((cls / len(classNames)) * 255 * 3)
                if escala >= 255*2:
                    R = 255
                    G = 255
                    B = escala - 255*2
                else:
                    if escala >= 255:
                        R = 255
                        G = escala - 255
                        B = 0
                    else:
                        R = escala
                        G = 0
                        B = 0

                # Dibuja el contenedor y clase
                cv2.rectangle(img, (x1, y1), (x2, y2), (R, G, B), 3)
                cv2.putText(img, track_id + ' ' + classNames[cls] , [x1, y1], cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, B), 2)

        # Muestra fotograma
        cv2.imshow('Vid', img)
    
    # Detenemos pulsado ESC
    if cv2.waitKey(20) == 27:
        break
  
# Libera el objeto de captura
vid.release()
# Destruye ventanas
cv2.destroyAllWindows()




0: 480x640 (no detections), 132.2ms
Speed: 6.7ms preprocess, 132.2ms inference, 9.5ms postprocess per image at shape (1, 3, 480, 640)

WARNING not enough matching points
0: 480x640 (no detections), 51.8ms
Speed: 2.0ms preprocess, 51.8ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 56.2ms
Speed: 2.0ms preprocess, 56.2ms inference, 0.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 47.1ms
Speed: 1.8ms preprocess, 47.1ms inference, 0.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 53.0ms
Speed: 1.4ms preprocess, 53.0ms inference, 0.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 50.4ms
Speed: 1.5ms preprocess, 50.4ms inference, 0.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 57.4ms
Speed: 2.3ms preprocess, 57.4ms inference, 0.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 47.

Intregración con seguimiento (tracking)
!!!!!!!!!Nota: he tenido que bajar a la versión de python 3.9.5 e instalar lap con pip install lap

In [3]:
# Carga del modelo
model = YOLO('yolo11n.pt') #Contenedores
#model = YOLO('yolov11n-seg.pt') #Máscaras
#model = YOLO('yolo11n-pose.pt')  #Pose

#Para un vídeo 
filename = "TGC23_PdH_C0056cut.mp4"
results = model.track(source=filename, show=True)  # BoT-SORT tracker (por defecto)
#results = model.track(source=filename, show=True, tracker="bytetrack.yaml")  # ByteTrack tracker

cv2.destroyAllWindows()


WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/375) c:\Users\ivanl\Desktop\Cosas de la uni\Año 2026 2027\VC\P1\P4\TGC23_PdH_C0056cut.mp4: 384x640 4 persons, 1 bird, 244.3ms
video 1/1 (frame 2/375) c:\Users\ivanl\Desktop\Cosas de la uni\Año 2026 2027\VC\P1\P4\TGC23_PdH_C0056cut.mp4: 384x640 4 persons, 76.9ms
video 1/1 (frame 3/375) c:\Users\ivanl\Desktop\Cosas de la uni\Año 2026 2027\VC\P1\P4\TGC23_PdH_C0056cut.mp4: 384x640 4 persons, 1 bird, 66.3ms
video 1/1 (frame 4/375) c:\Users\ivan

Tarea: Con el modelo general de YOLO detectar en el vídeo proporcionado vehículos y personas

In [4]:
class Tracked_Object:

    def __init__(self, id : int, type : int) -> None:
        self.id = id
        self.points = [] # los distintos puntos asociados a este objeto
        self.trayectory_x = 0
        self.trayectory_y = 0
        self.type = type

    def add_point(self, xy) -> None:
        x, y = xy[0], xy[1]
        self.points.append((x, y))

    def calculate_trayectory(self) -> None:
        self.trayectory_x = self.points[-1][0] - self.points[0][0]
        self.trayectory_y = self.points[-1][1] - self.points[0][1]

    def __str__(self) -> str:
        return f"ID:{self.id}, type:{self.type}"

class Image_Section: 

    def __init__(self, section_name : str) -> None:
        self.name = section_name
        self.class_names = {0:"person", 1:"bicycle", 2:"car", 3:"motorbike", 5:"bus", 7:"truck"}
        self.detections = {} # par clave-valor clase : cuenta

    def add_obj(self, obj : Tracked_Object) -> None:
        self.detections[obj.type] = self.detections.get(obj.type, 0) + 1

    def __str__(self):
        total = sum(self.detections.values())
        string = f"Esta sección de {self.name} contiene {total} objetos:\n"
        for key, count in self.detections.items():
            string += f"- {self.class_names.get(key, f'clase_{key}')} : {count}\n"
        return string



In [ ]:
#video
filename = "coches.MP4"
ANCHO_MAX = 1280

#clases que se van a detectar
classNames = ["person", "bicycle", "car", "motorbike", "bus", "truck"]
classCounter = {"person": 0, "bicycle": 0, "car": 0, "motorbike": 0, "bus": 0, "truck": 0}
classNames = {0: "person", 1: "bicycle", 2: "car", 3: "motorbike", 5: "bus", 7: "truck"}
uniqueObjects = {nombre: set() for nombre in classNames.values()}

#carga de los modelos
model = YOLO('yolo11n.pt')
last_known_position = {} 
objects = {} # clave-par id:obj
coches = cv2.VideoCapture(filename)

while(True):
    ret, frame = coches.read()

    if ret:
        alto, ancho = frame.shape[:2]
        if ancho > ANCHO_MAX:
            factor = ANCHO_MAX / ancho
            frame = cv2.resize(frame, (ANCHO_MAX, int(alto * factor)))
        alto, ancho = frame.shape[:2] 

        # Reiniciar contadores en cada frame
        classCounter = {nombre: 0 for nombre in classNames.values()}   

        #entrenar el modelo
        results = model.track(frame, persist=True, classes=[0,1,2,3,5,7], verbose=False)
        frame = results[0].plot()

        #recorrer resultados
        for result in results:
            for box in result.boxes:
                clase = int(box.cls.item())

                if clase not in classNames:
                    continue

                nombre = classNames[clase]
                classCounter[nombre] += 1

                if box.id is not None:
                    id_objeto = int(box.id.item())
                    uniqueObjects[nombre].add(id_objeto)
            
            ids = result.boxes.id.int().cpu().tolist()
            boxes_xywh = result.boxes.xywh.cpu().tolist()
            types = result.boxes.cls.int().cpu().tolist()

            for id, coords, type in zip(ids, boxes_xywh, types):
                if id not in objects:
                    objects[id] = Tracked_Object(id, type)
                objects[id].add_point(coords)

            #Dibujar contador
        objetos = []

        for nombre in classNames.values():
            actual = classCounter[nombre]
            total = len(uniqueObjects[nombre])

            objetos.append(f"{nombre}: {actual} (Total: {total})")

        fuente = cv2.FONT_HERSHEY_SIMPLEX
        escala = 0.55
        grosor = 2
        margen = 10
        separacion = 25

        # Calcular dimensiones del panel
        dimensiones = [
            cv2.getTextSize(texto, fuente, escala, grosor)[0]
            for texto in objetos
        ]

        ancho_panel = max(w for w, h in dimensiones) + 20
        alto_panel = len(objetos) * separacion + 15

        # Situar panel en esquina superior derecha
        x_panel = max(0, ancho - ancho_panel - margen)
        y_panel = margen

        # Dibujar fondo negro
        cv2.rectangle(
            frame,
            (x_panel, y_panel),
            (x_panel + ancho_panel, y_panel + alto_panel),
            (0, 0, 0),
            -1
        )

        # Dibujar cada lÃnea
        for i, texto in enumerate(objetos):

            x = x_panel + 10
            y = y_panel + 20 + i * separacion

            cv2.putText(
                frame,
                texto,
                (x, y),
                fuente,
                escala,
                (0, 255, 0),
                grosor,
                cv2.LINE_AA
            )

        cv2.imshow('Vid', frame)



    # Detenemos pulsado ESC
    if cv2.waitKey(20) == 27:
        break

coches.release()
cv2.destroyAllWindows()

lu = Image_Section("arriba izquierda")
ld = Image_Section("abajo izquierda")
ru = Image_Section("arriba derecha")
rd = Image_Section("abajo derecha")

for id in objects.keys():
    obj = objects[id]
    obj.calculate_trayectory()

    if obj.trayectory_x < 0 and obj.trayectory_y < 0:
        ld.add_obj(obj)
    elif obj.trayectory_x < 0 and obj.trayectory_y > 0:
        lu.add_obj(obj)
    elif obj.trayectory_x > 0 and obj.trayectory_y > 0:
        ru.add_obj(obj)
    else:
        rd.add_obj(obj)

print(objects)

print(lu)
print(ld)
print(ru)
print(rd)



{1: <__main__.Tracked_Object object at 0x000001F34D3F3460>, 2: <__main__.Tracked_Object object at 0x000001F34D3F3220>, 3: <__main__.Tracked_Object object at 0x000001F348C103A0>, 4: <__main__.Tracked_Object object at 0x000001F34887ECA0>, 5: <__main__.Tracked_Object object at 0x000001F350BB66A0>}
Esta sección de arriba izquierda contiene 2 objetos:
- bus : 1
- car : 1

Esta sección de abajo izquierda contiene 1 objetos:
- car : 1

Esta sección de arriba derecha contiene 0 objetos:

Esta sección de abajo derecha contiene 2 objetos:
- person : 2



In [4]:
model = YOLO(r'C:\Users\ivanl\Desktop\modelos\modelo_4\weights\best.pt')
results = model.track("coches.MP4", show=True, persist=True, classes=[0,1,2,3,5,7], verbose=False)

WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs



In [ ]:
from paddleocr import PaddleOCR
import matplotlib.pyplot as plt
import matplotlib.patches as patches

ocr = PaddleOCR(use_angle_cls=True, lang='es', enable_mkldnn=False)  # need to run only once to download and load model into memory
img_path = 'Test_manual.jpg'
result = ocr.predict(img_path)[0]

texts = result['rec_texts']
scores = result['rec_scores']
polys = result['rec_polys']  # one 4-point polygon per recognized text

print(texts, scores)

# Prepare the figure with the image
fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(plt.imread(img_path))
ax.axis('off')

for poly, text, score in zip(polys, texts, scores):
    print(f'\nText: {text}\nProbability: {score:.2f}\n'
          f'Box: {tuple(map(int, poly[0])), tuple(map(int, poly[2]))}')

    # Box around the detected text
    ax.add_patch(patches.Polygon(poly, closed=True, fill=False,
                                 edgecolor='lime', linewidth=2))

    # Recognized text, just above the top-left corner of the box
    x, y = poly[0]
    ax.text(x, y - 5, f'{text} ({score:.2f})',
            color='black', fontsize=9, va='bottom',
            bbox=dict(facecolor='lime', alpha=0.7, pad=1, edgecolor='none'))

plt.tight_layout()
plt.savefig(img_path.replace('.jpg', '_output_Paddle.jpg'), dpi=200, bbox_inches='tight')  # save the result
plt.show()

C:\Users\ivanl\AppData\Local\Temp\ipykernel_9952\675301837.py:5: DeprecationWarning: The parameter `use_angle_cls` has been deprecated and will be removed in the future. Please use `use_textline_orientation` instead.
  ocr = PaddleOCR(use_angle_cls=True, lang='es', enable_mkldnn=False)  # need to run only once to download and load model into memory
Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\ivanl\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\ivanl\.paddlex\official_models\UVDoc`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\ivanl\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
Creating model